In [1]:
!pip install kagglehub
!pip install rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=da239215216f84d3ab172e96b35aaa4be6119693d999248f5a13ba668ff192d2
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [2]:
!pip install transformers datasets rouge-score accelerate

In [3]:
import os
import kagglehub
import re
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BartForConditionalGeneration,
    BartTokenizer
)
from rouge_score import rouge_scorer
from torch.optim import AdamW

In [4]:
path = kagglehub.dataset_download(
    "gowrishankarp/newspaper-text-summarization-cnn-dailymail"
)

print("Dataset path:", path)
files = os.listdir(path)
print("Number of files:", len(files))

Using Colab cache for faster access to the 'newspaper-text-summarization-cnn-dailymail' dataset.
Dataset path: /kaggle/input/newspaper-text-summarization-cnn-dailymail
Number of files: 1


In [5]:
data_dir = os.path.join(path, "cnn_dailymail")

train_file = os.path.join(data_dir, "train.csv")
val_file   = os.path.join(data_dir, "validation.csv")
test_file  = os.path.join(data_dir, "test.csv")

print(os.listdir(data_dir))
import pandas as pd

train_df = pd.read_csv(train_file).head(5000)
val_df   = pd.read_csv(val_file).head(1000)
test_df  = pd.read_csv(test_file).head(1000)

['validation.csv', 'train.csv', 'test.csv']


In [6]:
import re
import unicodedata
import nltk
from nltk.tokenize import sent_tokenize

!pip install -q nltk

# Download tokenizer data
nltk.download("punkt")
nltk.download("punkt_tab") # Added to resolve LookupError

# -------------------------------
# BART-aware text cleaning
# -------------------------------
def clean_text_bart(text):
    if not isinstance(text, str):
        return ""

    # 1. Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # 2. Lowercase (optional but safe for CNN-style data)
    text = text.lower()

    # 3. Remove HTML/XML tags
    text = re.sub(r"<.*?>", " ", text)

    # 4. Remove URLs and emails
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)

    # 5. Normalize repeated punctuation
    text = re.sub(r"([.!?]){2,}", r"\1", text)
    text = re.sub(r"[,;:]{2,}", ",", text)

    # 6. Remove non-ASCII artifacts
    text = re.sub(r"[^\x00-\x7F]+", " ", text)

    # 7. Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # 8. Sentence-level deduplication
    sentences = sent_tokenize(text)
    seen = set()
    cleaned_sentences = []
    for s in sentences:
        if s not in seen and len(s.split()) > 3:
            cleaned_sentences.append(s)
            seen.add(s)

    return " ".join(cleaned_sentences)

# -------------------------------
# Apply cleaning
# -------------------------------
for df in [train_df, val_df, test_df]:
    df["article"] = df["article"].apply(clean_text_bart)
    df["highlights"] = df["highlights"].apply(clean_text_bart)

print("Text cleaning completed.")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Text cleaning completed.


In [7]:
MODEL_NAME = "facebook/bart-large-cnn"

tokenizer = BartTokenizer.from_pretrained(MODEL_NAME)
model = BartForConditionalGeneration.from_pretrained(MODEL_NAME)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model.to(DEVICE)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        

In [8]:
class CNNDailyMailDataset(Dataset):
    def __init__(self, articles, summaries, tokenizer, max_input=512, max_output=128):
        self.articles = articles
        self.summaries = summaries
        self.tokenizer = tokenizer
        self.max_input = max_input
        self.max_output = max_output

    def __len__(self):
        return len(self.articles)

    def __getitem__(self, idx):
        input_enc = tokenizer(
            self.articles[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_input,
            return_tensors="pt"
        )

        target_enc = tokenizer(
            self.summaries[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_output,
            return_tensors="pt"
        )

        return {
            "input_ids": input_enc["input_ids"].squeeze(),
            "attention_mask": input_enc["attention_mask"].squeeze(),
            "labels": target_enc["input_ids"].squeeze()
        }


In [9]:
train_dataset = CNNDailyMailDataset(
    train_df["article"].tolist(),
    train_df["highlights"].tolist(),
    tokenizer
)

val_dataset = CNNDailyMailDataset(
    val_df["article"].tolist(),
    val_df["highlights"].tolist(),
    tokenizer
)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=2)


In [10]:
optimizer = AdamW(model.parameters(), lr=2e-5)


In [11]:
def train_epoch(model, loader):
    model.train()
    total_loss = 0

    for batch in loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

for epoch in range(5):
    loss = train_epoch(model, train_loader)
    print(f"Epoch {epoch+1} | Train Loss: {loss:.4f}")

Epoch 1 | Train Loss: 0.8841
Epoch 2 | Train Loss: 0.6031
Epoch 3 | Train Loss: 0.4198
Epoch 4 | Train Loss: 0.2829
Epoch 5 | Train Loss: 0.1934


In [12]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def evaluate(model, df, n_samples=100):
    model.eval()
    scores = {"rouge1": [], "rouge2": [], "rougeL": []}

    for i in range(n_samples):
        article = df.iloc[i]["article"]
        reference = df.iloc[i]["highlights"]

        inputs = tokenizer(
            article,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(DEVICE)

        summary_ids = model.generate(
            **inputs,
            max_length=128,
            min_length=40,
            length_penalty=2.0,
            num_beams=4
        )

        generated = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        rouge = scorer.score(reference, generated)

        for k in scores:
            scores[k].append(rouge[k].fmeasure)

    return {k: sum(v)/len(v) for k, v in scores.items()}

rouge_scores = evaluate(model, test_df)
print(rouge_scores)

{'rouge1': 0.4116847493642784, 'rouge2': 0.17414742601357924, 'rougeL': 0.2723406276932353}


In [13]:
def show_examples(model, df, n_examples=3):
    model.eval()

    for i in range(n_examples):
        article = df.iloc[i]["article"]
        reference = df.iloc[i]["highlights"]

        inputs = tokenizer(
            article,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(DEVICE)

        summary_ids = model.generate(
            **inputs,
            max_length=128,
            min_length=40,
            length_penalty=2.0,
            num_beams=4
        )

        generated = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

        print("=" * 80)
        print(f"Example {i + 1}")
        print("-" * 80)
        print("ARTICLE:\n", article[:1000], "...\n")
        print("REFERENCE SUMMARY:\n", reference, "\n")
        print("GENERATED SUMMARY:\n", generated, "\n")
show_examples(model, test_df, n_examples=5)

Example 1
--------------------------------------------------------------------------------
ARTICLE:
 ever noticed how plane seats appear to be getting smaller and smaller? with increasing numbers of people taking to the skies, some experts are questioning if having such packed out planes is putting passengers at risk. they say that the shrinking space on aeroplanes is not only uncomfortable - it's putting our health and safety in danger. more than squabbling over the arm rest, shrinking space on planes putting our health and safety in danger? this week, a u.s consumer advisory group set up by the department of transportation said at a public hearing that while the government is happy to set standards for animals flying on planes, it doesn't stipulate a minimum amount of space for humans. 'in a world where animals have more rights to space and food than humans,' said charlie leocha, consumer representative on the committee. 'it is time that the dot and faa take a stand for humane treatm